In [1]:
%pip install sentence-transformers rank-bm25 transformers torch numpy

In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

In [3]:
corpus = [
    "Transformers use self-attention mechanisms to weigh the importance of different words in a sequence.",
    "Self-attention allows each token to attend to all other tokens in the input sequence.",
    "Backpropagation is used to compute gradients for neural network training.",
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates for faster convergence.",
    "Overfitting occurs when a model learns noise instead of general patterns.",
    "Dropout is a regularization technique that prevents overfitting in neural networks.",
    "BERT is a transformer-based model pretrained using masked language modeling.",
    "ReLU is an activation function that outputs zero for negative inputs.",
    "Batch normalization stabilizes learning by normalizing layer inputs.",
    "The term 'attention is all you need' refers to the transformer architecture paper.",
    "Tokenization converts raw text into smaller units like subwords or tokens."
]

In [5]:
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

generator = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    max_new_tokens=100
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaF

In [6]:
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        # SBERT
        self.doc_vecs = bi_encoder.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = self.doc_vecs / np.linalg.norm(self.doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query, top_k=5):
        # BM25
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # SBERT
        query_vec = bi_encoder.encode([query], convert_to_numpy=True)[0]
        query_vec = query_vec / np.linalg.norm(query_vec)
        sbert_scores = self.doc_vecs @ query_vec
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        results = []

        for i in range(len(self.corpus)):
            bm_rank = np.where(bm25_ranks == i)[0][0]
            sb_rank = np.where(sbert_ranks == i)[0][0]

            rrf = (1 / (self.k + bm_rank)) + (1 / (self.k + sb_rank))

            results.append({
                "doc_id": i,
                "rrf_score": rrf,
                "bm25_rank": int(bm_rank),
                "sbert_rank": int(sb_rank),
                "text": self.corpus[i]
            })

        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

In [7]:
def rerank(query, candidates, top_k=3):
    pairs = [[query, c["text"]] for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["cross_score"] = float(scores[i])

    ranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return ranked[:top_k]

In [8]:
def hyde_expand(query):
    prompt = f"""
    Write a short factual paragraph answering the question:
    {query}
    """
    result = generator(prompt)[0]["generated_text"]
    return result

In [9]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    # Step 1: HyDE
    expanded_query = hyde_expand(user_query)

    # Step 2: Retrieval
    candidates = retriever.retrieve(expanded_query, top_k=5)

    # Step 3: Re-ranking
    reranked = rerank(user_query, candidates, top_k=3)

    # Step 4: Context
    context = "\n".join([doc["text"] for doc in reranked])

    # Step 5: Final Answer
    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {user_query}
    """

    answer = generator(prompt)[0]["generated_text"]
    return answer

In [12]:
def naive_rag(query):
    vec = bi_encoder.encode([query], convert_to_numpy=True)[0]
    vec = vec / np.linalg.norm(vec)

    scores = retriever.doc_vecs @ vec
    idx = np.argsort(scores)[::-1][:1]

    return corpus[idx[0]]

In [13]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is overfitting?"
]

for q in queries:
    print("="*60)
    print("Query:", q)

    naive = naive_rag(q)
    advanced = advanced_rag(q)

    print("\nNaive RAG Top Doc:")
    print(naive)

    print("\nAdvanced RAG Answer:")
    print(advanced)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: how do transformers encode meaning?


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Naive RAG Top Doc:
Transformers use self-attention mechanisms to weigh the importance of different words in a sequence.

Advanced RAG Answer:

    Answer the question using the context below.

    Context:
    Transformers use self-attention mechanisms to weigh the importance of different words in a sequence.
The term 'attention is all you need' refers to the transformer architecture paper.
BERT is a transformer-based model pretrained using masked language modeling.

    Question:
    how do transformers encode meaning?
     hand by the parent: [f]. Each generation shows for specific needs an expression; formula[
Query: optimization techniques for training


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Naive RAG Top Doc:
Gradient descent is an optimization algorithm used to minimize loss functions.

Advanced RAG Answer:

    Answer the question using the context below.

    Context:
    Gradient descent is an optimization algorithm used to minimize loss functions.
Adam optimizer combines momentum and adaptive learning rates for faster convergence.
Transformers use self-attention mechanisms to weigh the importance of different words in a sequence.

    Question:
    optimization techniques for training
    
Query: what is overfitting?


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Naive RAG Top Doc:
Overfitting occurs when a model learns noise instead of general patterns.

Advanced RAG Answer:

    Answer the question using the context below.

    Context:
    Overfitting occurs when a model learns noise instead of general patterns.
Dropout is a regularization technique that prevents overfitting in neural networks.
The term 'attention is all you need' refers to the transformer architecture paper.

    Question:
    what is overfitting?
    


| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|------|------------------|---------------------|--------------------|
| how do transformers encode meaning? | Transformers use self-attention... | More detailed explanation | Yes |
| optimization techniques for training | Gradient descent... | Includes Adam + explanation | Yes |
| what is overfitting? | Overfitting occurs... | More contextual explanation | Yes |